### Loading all silver tables

In [0]:
meters = spark.table("smartmeter.silver.meters")

readings = spark.table("smartmeter.silver.readings")

registers = spark.table("smartmeter.silver.registers")

alarms = spark.table("smartmeter.silver.alarms")

areas = spark.table("smartmeter.silver.areas")

distributors = spark.table("smartmeter.silver.distributors")


In [0]:
distributors.show(10)

## What meters do we have? (gold_meter_overview)

In [0]:
gold_meter_overview = (
    meters.join(areas, on="area_id", how="left")\
        .join(distributors, on="distributor_id", how="left")\
            .select(meters["meter_id"],\
                meters["serial_number"],
                meters["meter_type"],
                meters["manufacturer"],
                meters["model"],
                meters["protocol"],
                meters["rated_capacity"],
                meters["capacity_unit"],
                meters["install_date"],
                meters["status"],
                meters["area_id"],
                areas["name"].alias("area_name"),
                areas["state"],
                meters["distributor_id"],
                distributors["name"].alias("distributor_name"),
                distributors["license_region"]
            )
        )

display(gold_meter_overview)

### saving gold_meter_overview to gold schema

In [0]:
gold_meter_overview.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("smartmeter.gold.meter_overview")


## Are the meters healthy/online? (gold_meter_health)

In [0]:
from pyspark.sql.functions import col, count, row_number
from pyspark.sql.window import Window


# 1. Get latest reading for each meter
reading_window = (Window.partitionBy("meter_id") .orderBy(col("timestamp")\
    .desc()))

latest_readings = (readings.withColumn("rn", row_number()\
    .over(reading_window)).filter(col("rn") == 1).drop("rn"))

# 2. Get latest alarm for each meter
alarm_window = (Window.partitionBy("meter_id").orderBy(col("timestamp")\
    .desc()))

latest_alarms = (alarms.withColumn("rn", row_number().over(alarm_window))\
    .filter(col("rn") == 1).drop("rn"))

# 3. Count total alarms for each meter
alarm_counts = (alarms.groupBy("meter_id")\
    .agg(count("*").alias("alarm_count")))

# 4. Build Gold Meter Health
gold_meter_health = (meters
    .join(
        areas,
        meters["area_id"] == areas["area_id"],
        "left"
    )
    .join(
        distributors,
        areas["distributor_id"] == distributors["distributor_id"],
        "left"
    )
    .join(
        latest_readings,
        meters["meter_id"] == latest_readings["meter_id"],
        "left"
    )
    .join(
        latest_alarms,
        meters["meter_id"] == latest_alarms["meter_id"],
        "left"
    )
    .join(
        alarm_counts,
        meters["meter_id"] == alarm_counts["meter_id"],
        "left"
    )
    .select(
        meters["meter_id"],
        meters["serial_number"],
        meters["meter_type"],

        areas["name"].alias("area_name"),
        areas["state"],

        meters["status"],

        latest_readings["timestamp"].alias("last_reading_time"),
        latest_readings["quality"].alias("last_reading_quality"),

        latest_alarms["timestamp"].alias("last_alarm_time"),
        latest_alarms["type"].alias("last_alarm_type"),
        latest_alarms["severity"].alias("last_alarm_severity"),

        alarm_counts["alarm_count"]
    )
)


display(gold_meter_health)

### saving gold_meter_health to gold schema


In [0]:
gold_meter_health.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("smartmeter.gold.meter_health")

## What problems are occurring? (gold_alarm_summary)

In [0]:
from pyspark.sql.functions import (col,count,sum,min,max)

gold_alarm_summary = (alarms
    .join(
        meters,
        alarms["meter_id"] == meters["meter_id"],
        "left")
    .join(
        areas,
        meters["area_id"] == areas["area_id"],
        "left")
    .groupBy(
        alarms["type"].alias("alarm_type"),
        alarms["severity"],
        areas["name"].alias("area_name"),
        areas["state"],
        meters["meter_type"])
    .agg(
        count("*").alias("alarm_count"),
        sum(
            col("acknowledged").cast("int")
        ).alias("acknowledged_count"),
        sum(
            (~col("acknowledged")).cast("int")
        ).alias("unacknowledged_count"),
        min(alarms["timestamp"]).alias("first_alarm_time"),
        max(alarms["timestamp"]).alias("last_alarm_time")
    )
)

display(gold_alarm_summary)

### Saving gold_alarm_summary to gold

In [0]:
gold_alarm_summary.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("smartmeter.gold.alarm_summary")


###  How much electricity/water/gas is being used?(utility_usage)

In [0]:
from pyspark.sql.functions import col

gold_utility_usage = (
    readings
    .join(
        registers,
        readings["register_id"] == registers["register_id"],
        "left"
    )
    .join(
        meters,
        readings["meter_id"] == meters["meter_id"],
        "left"
    )
    .join(
        areas,
        meters["area_id"] == areas["area_id"],
        "left"
    )
    .filter(
        (
            (meters["meter_type"] == "electricity") &
            (registers["parameter_name"] == "Active power")
        )
        |
        (
            (meters["meter_type"] == "water") &
            (registers["parameter_name"].isin(
                "Flow rate",
                "Cumulative volume"
            ))
        )
        |
        (
            (meters["meter_type"] == "gas") &
            (registers["parameter_name"].isin(
                "Flow rate",
                "Cumulative volume"
            ))
        )
    )
    .select(
        readings["meter_id"],
        meters["meter_type"],
        areas["name"].alias("area_name"),
        readings["timestamp"],
        registers["parameter_name"],
        readings["value"],
        registers["unit"],
        readings["quality"],
        meters["manufacturer"],
        meters["model"]
    )
)

display(gold_utility_usage)

## saving gold_utility_usage to gold

In [0]:
gold_utility_usage.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("smartmeter.gold.utility_usage")

### Is the electricity quality good? (power_quality)

In [0]:
from pyspark.sql.functions import col

gold_power_quality = (readings
    .join(
        registers,
        readings["register_id"] == registers["register_id"],
        "left"
    )
    .join(
        meters,
        readings["meter_id"] == meters["meter_id"],
        "left"
    )
    .join(
        areas,
        meters["area_id"] == areas["area_id"],
        "left"
    )
    .filter(
        (meters["meter_type"] == "electricity") &
        (registers["parameter_name"].isin(
            "Voltage (L1)",
            "Current (L1)",
            "Active power"
        ))
    )
    .select(
        readings["meter_id"],
        meters["serial_number"],
        areas["name"].alias("area_name"),
        readings["timestamp"],
        registers["parameter_name"],
        readings["value"],
        registers["unit"],
        readings["quality"],
        meters["manufacturer"],
        meters["model"]
    )
)

display(gold_power_quality)

### saving gold_power_quality to gold schema

In [0]:
gold_power_quality.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("smartmeter.gold.power_quality")

### Can we trust our data? (data_quality)

In [0]:
from pyspark.sql.functions import (col,count,sum,min,max,round)

# 1. Total readings, good readings and bad readings
reading_quality = (readings
    .groupBy("meter_id")
    .agg(
        count("*").alias("total_readings"),
        sum(
            (col("quality") == "GOOD").cast("int")).alias("good_readings"),
        sum(
            (col("quality") != "GOOD").cast("int")).alias("bad_readings"),
        min("timestamp").alias("first_reading_time"),
        max("timestamp").alias("last_reading_time")
    )
)

# 2. Find duplicate readings
duplicate_readings = (
    readings
    .groupBy("meter_id","register_id","timestamp")
    .count()
    .filter(col("count") > 1)
    .groupBy("meter_id")
    .agg(sum(col("count") - 1).alias("duplicate_readings"))
)

# 3. Build Gold Data Quality table
gold_data_quality = (meters
    .join(reading_quality,
        meters["meter_id"] == reading_quality["meter_id"],"left")
    .join(
        duplicate_readings,
        meters["meter_id"] == duplicate_readings["meter_id"],"left")
    .join(
        areas,
        meters["area_id"] == areas["area_id"],"left")
    .select(
        meters["meter_id"],
        meters["meter_type"],
        areas["name"].alias("area_name"),

        reading_quality["total_readings"],
        reading_quality["good_readings"],
        reading_quality["bad_readings"],

        duplicate_readings["duplicate_readings"],

        reading_quality["first_reading_time"],
        reading_quality["last_reading_time"]
    )
)


display(gold_data_quality)

### Adding completeness percentage

In [0]:
from pyspark.sql.functions import coalesce, lit

gold_data_quality = gold_data_quality.withColumn(
    "data_completeness_pct",
    round(col("good_readings") / col("total_readings") * 100,2))

display(gold_data_quality)

### saving gold_data_quality to gold schema

In [0]:
gold_data_quality.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("smartmeter.gold.data_quality")